In [1]:
import pandas as pd
import torch
from  torch.optim import AdamW, Adam, SGD
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments, get_linear_schedule_with_warmup
from datasets import Dataset
from sklearn.metrics import accuracy_score
import gc
from math import ceil
from transformers import EarlyStoppingCallback
from utils.config import QUESTION_TYPES

In [2]:
new_dataset_path= 'Model_dataset/real_world_data.csv'
old_dataset_path= 'Model_dataset/synthetic_question_ans_data-v2.csv'

# q type classifer,
q_type_model_path= 'model/fine_tuned_question_classifier_model_lite-default'
q_type_model_result= '.temp/model_results/q_types_model_lite_results'
q_type_model= '.temp/model/fine_tuned_question_classifier_model_lite'



# Question type classsifier

### Preprocessing

In [3]:
data_ratio= {
    "old":20,
    "new":80
    }

new_data_df= pd.read_csv(new_dataset_path)
new_data_df= new_data_df[["question", "question_type"]]
new_data_df.info()

old_data_df= pd.read_csv(old_dataset_path)
old_data_df= old_data_df[["question", "question_type"]]
old_data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1882 entries, 0 to 1881
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       1882 non-null   object
 1   question_type  1882 non-null   object
dtypes: object(2)
memory usage: 29.5+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1644 entries, 0 to 1643
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       1644 non-null   object
 1   question_type  1644 non-null   object
dtypes: object(2)
memory usage: 25.8+ KB


In [4]:
columns_in_new_df= new_data_df["question_type"].unique()
print(f"columns_in_new_df :{columns_in_new_df}")

columns_in_old_df= old_data_df["question_type"].unique()
print(f"columns_in_old_df :{columns_in_old_df}")

columns_in_new_df :['availability' 'personal_information' 'current_ctc' 'education'
 'working_experience' 'expected_ctc' 'others' 'skills']
columns_in_old_df :['current_ctc' 'expected_ctc' 'personal_information' 'education'
 'working_experience' 'skills' 'availability' 'others']


In [5]:
new_data_df.drop_duplicates(inplace= True)
new_data_df.info()

old_data_df.drop_duplicates(inplace= True)
old_data_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1866 entries, 0 to 1881
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       1866 non-null   object
 1   question_type  1866 non-null   object
dtypes: object(2)
memory usage: 43.7+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 1634 entries, 0 to 1643
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       1634 non-null   object
 1   question_type  1634 non-null   object
dtypes: object(2)
memory usage: 38.3+ KB


#### combaing new and old data

In [6]:
new_data_len= len(new_data_df)
total_len= ceil(new_data_len/(data_ratio["new"]/100))
old_data_len= ceil(total_len- new_data_len)
print(total_len)
old_data_len

2333


467

In [7]:
temp_df= pd.DataFrame()
while True:
    temp_df= old_data_df.sample(old_data_len)
    columns_in_old_df= temp_df["question_type"].unique()

    if set(columns_in_old_df)== set(columns_in_new_df):
        break
temp_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 467 entries, 42 to 496
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       467 non-null    object
 1   question_type  467 non-null    object
dtypes: object(2)
memory usage: 10.9+ KB


In [8]:
df= pd.concat([new_data_df, temp_df], ignore_index=True)
df.drop_duplicates(inplace= True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2331 entries, 0 to 2332
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       2331 non-null   object
 1   question_type  2331 non-null   object
dtypes: object(2)
memory usage: 54.6+ KB


#### adding labels

In [9]:
# adding labels
label_mapping = {key: index for index, key in enumerate(QUESTION_TYPES)}
df['label'] = df['question_type'].map(label_mapping)

# droping unused column
df.drop('question_type', axis=1,  inplace= True)
df.head()

,question,label
0,Are u okay with Bhubaneshwar Location for Walk...,6
1,What is your current notice period in your com...,6
2,What is your notice period? in days,6
3,Are you okay with Attending a walk in/Offline ...,6
4,Can you join within 7 days?,6


In [10]:
df= df.sample(frac=1).reset_index(drop=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2331 entries, 0 to 2330
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   question  2331 non-null   object
 1   label     2331 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 36.6+ KB


In [11]:
df.head(5)

,question,label
0,How many years of work experience do you have ...,5
1,Are you comfortable with hybrid working 3 days...,6
2,Your education level?,3
3,How many years of work experience do you have ...,5
4,Do you have an understanding about different v...,7


### Retraing Preparations:

In [12]:
#convert to hugging face dataset
dataset= Dataset.from_pandas(df)

#Split the data into train and test sets (80-20 split)
dataset_split = dataset.train_test_split(test_size=0.2)

# Access train and test splits
train_dataset = dataset_split['train']
test_dataset = dataset_split['test']

In [13]:
tokenizer = DistilBertTokenizerFast.from_pretrained(q_type_model_path)

In [14]:
def tokenize_function(examples):
    return tokenizer(examples['question'], padding= "max_length", truncation=True)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set the format to PyTorch tensors
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

Map:   0%|          | 0/1864 [00:00<?, ? examples/s]

Map:   0%|          | 0/467 [00:00<?, ? examples/s]

In [15]:
# Mapping lebel and id
id2label = {v: k for k, v in label_mapping.items()}  # Map IDs to label names
label2id = {k: v for k, v in label_mapping.items()}  # Map label names to IDs

In [16]:
total_training_steps = (len(train_dataset) // (16 * 2)) * 4  # Example calculation: adjust as needed
warmup_steps = int(0.1 * total_training_steps)

# Retraning

# using default optimizer

In [17]:
# Loading the pre trained model
model = DistilBertForSequenceClassification.from_pretrained(
        q_type_model_path,
        num_labels= len(columns_in_old_df),
        ignore_mismatched_sizes=True,  # Allows resizing of classification head
        id2label=id2label,
        label2id=label2id,
    )
print(model.config.num_labels)  # Should print 7

8


In [18]:
training_args = TrainingArguments(
    output_dir=q_type_model_result + "-default",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    num_train_epochs= 50,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
)

In [19]:
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=train_dataset,         # The training dataset
    eval_dataset=test_dataset,           # The test dataset
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.predictions.argmax(axis=-1), p.label_ids)}  # Compute accuracy during eval
)


# Clearing memory before start traning
torch.cuda.empty_cache()
gc.collect()

# Start training
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.400400,0.275483,0.920771
2,0.262000,0.250743,0.933619
3,0.109300,0.277957,0.920771
4,0.103400,0.306529,0.918630


TrainOutput(global_step=236, training_loss=0.2054272473363553, metrics={'train_runtime': 7066.1519, 'train_samples_per_second': 13.19, 'train_steps_per_second': 0.41, 'total_flos': 987782607273984.0, 'train_loss': 0.2054272473363553, 'epoch': 4.0})

- seems like epoch 3 will be best for prediction

### Model evaluation and Saving

In [20]:
evaluation_results = trainer.evaluate()
evaluation_results

{'eval_loss': 0.2507428526878357,
 'eval_accuracy': 0.9336188436830836,
 'eval_runtime': 160.256,
 'eval_samples_per_second': 2.914,
 'eval_steps_per_second': 0.187,
 'epoch': 4.0}

In [21]:
trainer.save_model(q_type_model+"-default")
tokenizer.save_pretrained(q_type_model+"-default")

('.temp/model/fine_tuned_question_classifier_model_lite-default/tokenizer_config.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-default/special_tokens_map.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-default/vocab.txt',
 '.temp/model/fine_tuned_question_classifier_model_lite-default/added_tokens.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-default/tokenizer.json')

In [22]:
del trainer, model

# Using AdamW optimiser with linear scheduler with warmup

In [23]:
# Loading the pre trained model
model = DistilBertForSequenceClassification.from_pretrained(
        q_type_model_path,
        num_labels= len(columns_in_old_df),
        ignore_mismatched_sizes=True,  # Allows resizing of classification head
        id2label=id2label,
        label2id=label2id,
    )
print(model.config.num_labels)  # Should print 7

8


In [24]:
training_args = TrainingArguments(
    output_dir=q_type_model_result + "-AdamW",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    num_train_epochs= 50,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
)

In [25]:
# AdamW optimizer is automatically used by Hugging Face, but you can explicitly define it
optimizer = AdamW(model.parameters(), lr=5e-5, eps=1e-8)

# Define the linear scheduler with warmup
total_steps = len(train_dataset) * training_args.num_train_epochs // training_args.per_device_train_batch_size
warmup_steps = int(total_steps * 0.1)  # 10% of total steps for warmup
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

In [26]:
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=train_dataset,         # The training dataset
    eval_dataset=test_dataset,           # The test dataset
    optimizers=(optimizer, lr_scheduler),   # Pass optimizer and scheduler as a tuple
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.predictions.argmax(axis=-1), p.label_ids)}  # Compute accuracy during eval
)


# Clearing memory before start traning
torch.cuda.empty_cache()
gc.collect()

# Start training
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.479500,0.353222,0.890792
2,0.323800,0.279048,0.910064
3,0.178800,0.297719,0.905782
4,0.171300,0.297957,0.918630


TrainOutput(global_step=236, training_loss=0.2913241315696199, metrics={'train_runtime': 6999.3197, 'train_samples_per_second': 13.316, 'train_steps_per_second': 0.414, 'total_flos': 987782607273984.0, 'train_loss': 0.2913241315696199, 'epoch': 4.0})

In [27]:
evaluation_results = trainer.evaluate()
evaluation_results

{'eval_loss': 0.27904775738716125,
 'eval_accuracy': 0.9100642398286938,
 'eval_runtime': 159.2781,
 'eval_samples_per_second': 2.932,
 'eval_steps_per_second': 0.188,
 'epoch': 4.0}

In [28]:
trainer.save_model(q_type_model+"-AdamW")
tokenizer.save_pretrained(q_type_model+"-AdamW")

('.temp/model/fine_tuned_question_classifier_model_lite-AdamW/tokenizer_config.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-AdamW/special_tokens_map.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-AdamW/vocab.txt',
 '.temp/model/fine_tuned_question_classifier_model_lite-AdamW/added_tokens.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-AdamW/tokenizer.json')

In [29]:
del trainer, model


# Using Adam

In [30]:
# Loading the pre trained model
model = DistilBertForSequenceClassification.from_pretrained(
        q_type_model_path,
        num_labels= len(columns_in_old_df),
        ignore_mismatched_sizes=True,  # Allows resizing of classification head
        id2label=id2label,
        label2id=label2id,
    )
print(model.config.num_labels)  # Should print 7

8


In [31]:
training_args = TrainingArguments(
    output_dir=q_type_model_result + "-Adam",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    num_train_epochs= 50,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
)

In [32]:
# AdamW optimizer is automatically used by Hugging Face, but you can explicitly define it
optimizer = Adam(model.parameters(), lr=3e-5)

# Define the linear scheduler with warmup
total_steps = len(train_dataset) * training_args.num_train_epochs // training_args.per_device_train_batch_size
warmup_steps = int(total_steps * 0.1)  # 10% of total steps for warmup
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

In [33]:
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=train_dataset,         # The training dataset
    eval_dataset=test_dataset,           # The test dataset
    optimizers=(optimizer, lr_scheduler),   # Pass optimizer and scheduler as a tuple
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.predictions.argmax(axis=-1), p.label_ids)}  # Compute accuracy during eval
)


# Clearing memory before start traning
torch.cuda.empty_cache()
gc.collect()

# Start training
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.521100,0.392427,0.888651
2,0.361700,0.291868,0.903640
3,0.198400,0.292555,0.901499
4,0.191100,0.269146,0.918630
5,0.167800,0.262702,0.922912
6,0.112900,0.281236,0.916488
7,0.106200,0.317444,0.912206


TrainOutput(global_step=413, training_loss=0.24268861396405078, metrics={'train_runtime': 12203.6904, 'train_samples_per_second': 7.637, 'train_steps_per_second': 0.238, 'total_flos': 1728619562729472.0, 'train_loss': 0.24268861396405078, 'epoch': 7.0})

In [34]:
evaluation_results = trainer.evaluate()
evaluation_results

{'eval_loss': 0.262702077627182,
 'eval_accuracy': 0.9229122055674518,
 'eval_runtime': 157.7838,
 'eval_samples_per_second': 2.96,
 'eval_steps_per_second': 0.19,
 'epoch': 7.0}

In [35]:
trainer.save_model(q_type_model+"-Adam")
tokenizer.save_pretrained(q_type_model+"-Adam")

('.temp/model/fine_tuned_question_classifier_model_lite-Adam/tokenizer_config.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-Adam/special_tokens_map.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-Adam/vocab.txt',
 '.temp/model/fine_tuned_question_classifier_model_lite-Adam/added_tokens.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-Adam/tokenizer.json')

In [36]:
del trainer, model


# Using SGD

In [37]:
# Loading the pre trained model
model = DistilBertForSequenceClassification.from_pretrained(
        q_type_model_path,
        num_labels= len(columns_in_old_df),
        ignore_mismatched_sizes=True,  # Allows resizing of classification head
        id2label=id2label,
        label2id=label2id,
    )
print(model.config.num_labels)  # Should print 7

8


In [38]:
training_args = TrainingArguments(
    output_dir=q_type_model_result + "-SGD",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    num_train_epochs= 50,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
)

In [39]:
# AdamW optimizer is automatically used by Hugging Face, but you can explicitly define it
optimizer = SGD(model.parameters(), lr=0.01, momentum=0.9)

# Define the linear scheduler with warmup
total_steps = len(train_dataset) * training_args.num_train_epochs // training_args.per_device_train_batch_size
warmup_steps = int(total_steps * 0.1)  # 10% of total steps for warmup
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

In [40]:
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=train_dataset,         # The training dataset
    eval_dataset=test_dataset,           # The test dataset
    optimizers=(optimizer, lr_scheduler),   # Pass optimizer and scheduler as a tuple
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.predictions.argmax(axis=-1), p.label_ids)}  # Compute accuracy during eval
)


# Clearing memory before start traning
torch.cuda.empty_cache()
gc.collect()

# Start training
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.499900,0.388546,0.884368
2,0.367300,0.299589,0.901499
3,0.213200,0.304361,0.903640
4,0.218000,0.284871,0.918630
5,0.202400,0.273154,0.912206
6,0.173600,0.258681,0.922912
7,0.147200,0.290725,0.912206
8,0.125800,0.269467,0.925054


TrainOutput(global_step=472, training_loss=0.24607796632384848, metrics={'train_runtime': 13726.9171, 'train_samples_per_second': 6.79, 'train_steps_per_second': 0.211, 'total_flos': 1975565214547968.0, 'train_loss': 0.24607796632384848, 'epoch': 8.0})

In [41]:
evaluation_results = trainer.evaluate()
evaluation_results

{'eval_loss': 0.25868141651153564,
 'eval_accuracy': 0.9229122055674518,
 'eval_runtime': 155.5622,
 'eval_samples_per_second': 3.002,
 'eval_steps_per_second': 0.193,
 'epoch': 8.0}

In [42]:
trainer.save_model(q_type_model+"-SGD")
tokenizer.save_pretrained(q_type_model+"-SGD")

('.temp/model/fine_tuned_question_classifier_model_lite-SGD/tokenizer_config.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-SGD/special_tokens_map.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-SGD/vocab.txt',
 '.temp/model/fine_tuned_question_classifier_model_lite-SGD/added_tokens.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-SGD/tokenizer.json')

In [43]:
del trainer, model


: 